In [ ]:
import os
import glob
import pandas as pd
from tensorboard.backend.event_processing.event_accumulator import EventAccumulator
import json
from pathlib import Path
import numpy as np
from tqdm.notebook import tqdm

In [ ]:
# Base directory containing all expression runs
base_dir = "/runs/expression"

# Function to extract scores from TensorBoard event files
def extract_scores_from_events(event_file_path, run_name):
	"""Extract all scalar values at a specific step from TensorBoard event file."""
	try:
		ea = EventAccumulator(event_file_path)
		ea.Reload()
		
		scores = []
		for tag in ea.Tags()['scalars']:
			events = ea.Scalars(tag)
			# Find events at or closest to target step
			step_values = [(event.step, event.value) for event in events]
						
			for step, value in step_values:
				if step > 100: # skip test runs
					scores.append({"step": step, "value": value, "run_name": run_name, "metric": tag})
		return scores
	except Exception as e:
		print(f"Error processing {event_file_path}: {e}")
		return {}

# Main function to traverse all subdirectories and collect scores
def collect_all_scores(base_dir):
    """Traverse all subdirectories recursively and collect scores at specified step."""
    all_results = []
    all_dirs = list(os.walk(base_dir))
    for ind, (root, dirs, files) in enumerate(all_dirs):
        # Find TensorBoard event files in this directory
        event_files = glob.glob(os.path.join(root, "events.out.tfevents*"))
        if not event_files:
            continue
        print (ind, " of ",len(all_dirs))            
        # print(f"Processing: {root}")
        for event_file in event_files:
            # Only keep the part of the path after base_dir in run_name
            rel_run_name = os.path.relpath(root, base_dir)
            scores = extract_scores_from_events(event_file, run_name=rel_run_name)
            if scores:
                all_results.extend(scores)
                # print(f"  Found {len(scores)} metrics")
            else:
                pass
                # print(f"  No scores found")
    return all_results
	
# Execute the collection
print(f"Starting to collect scores from all subdirectories...")
results = collect_all_scores(base_dir)

print(f"\nCollected data from {len(results)} runs")

# Convert to DataFrame for easier analysis
if results:
	df = pd.DataFrame.from_records(results)
	print(f"\nDataFrame shape: {df.shape}")
	print("\nColumns:", df.columns.tolist())
	
	# Display first few rows
	print("\nFirst few results:")
	display(df.head())
	
	# Save to CSV
	output_file = f"expression_scores.csv"
	df.to_csv(output_file, index=False, compression="gzip")
	print(f"\nResults saved to {output_file}")
else:
	print("No results found!")

In [12]:
df["run_name"].unique()

array(['/mnt/10tb/home/vsfishman/DNALM/runs/expression/20250413-202948',
       '/mnt/10tb/home/vsfishman/DNALM/runs/expression/20250419-222413',
       '/mnt/10tb/home/vsfishman/DNALM/runs/expression/20250413-220701'],
      dtype=object)

In [13]:
df.query("run_name == '/mnt/10tb/home/vsfishman/DNALM/runs/expression/20250413-202948'")

,step,value,run_name,metric
0,250,11.555820,/mnt/10tb/home/vsfishman/DNALM/runs/expression...,loss/iterations/train
1,500,11.434798,/mnt/10tb/home/vsfishman/DNALM/runs/expression...,loss/iterations/train
2,750,11.261512,/mnt/10tb/home/vsfishman/DNALM/runs/expression...,loss/iterations/train
3,1000,11.167288,/mnt/10tb/home/vsfishman/DNALM/runs/expression...,loss/iterations/train
4,1250,10.246181,/mnt/10tb/home/vsfishman/DNALM/runs/expression...,loss/iterations/train
...,...,...,...,...
63,1000,0.000000,/mnt/10tb/home/vsfishman/DNALM/runs/expression...,patience/samples
64,2000,1.000000,/mnt/10tb/home/vsfishman/DNALM/runs/expression...,patience/samples
65,3000,0.000000,/mnt/10tb/home/vsfishman/DNALM/runs/expression...,patience/samples
66,4000,1.000000,/mnt/10tb/home/vsfishman/DNALM/runs/expression...,patience/samples


In [24]:
_ = df.query("run_name == '/mnt/10tb/home/vsfishman/DNALM/runs/expression/20250413-202948'")
_[_.metric.str.contains("iterations")].head(2)

,step,value,run_name,metric
0,250,11.555820,/mnt/10tb/home/vsfishman/DNALM/runs/expression...,loss/iterations/train
1,500,11.434798,/mnt/10tb/home/vsfishman/DNALM/runs/expression...,loss/iterations/train


In [25]:
_[_.metric.str.contains("samples")].head(2)

,step,value,run_name,metric
6,1000,11.555820,/mnt/10tb/home/vsfishman/DNALM/runs/expression...,loss/samples/train
7,2000,11.434798,/mnt/10tb/home/vsfishman/DNALM/runs/expression...,loss/samples/train


In [30]:
df = pd.read_csv("expression_scores.csv", compression="gzip")
df.head(2)

,step,value,run_name,metric
0,328100,23.744200,resume_20250508-085434/model_best.pth/20250517...,loss/iterations/train
1,328200,12.728351,resume_20250508-085434/model_best.pth/20250517...,loss/iterations/train


In [53]:
sel = df.query("run_name == 'upsample/ctloss/20250622-180659'")
_met = [i for i in sel.metric.unique() if ("valid" in i) and ("v1" in i or "loss" in i) and ("samples" in i) and not ("_4_" in i)]
sel = sel.query("metric in @_met").dropna(subset=["value"])
results = sel.groupby("metric").apply(lambda x: x.loc[x["value"].idxmin()] if x["metric"].str.contains("loss").any() else x.loc[x["value"].idxmax()])
results.reset_index(drop=True).sort_values("metric", ascending=False)

,step,value,run_name,metric
6,12384000,0.339130,upsample/ctloss/20250622-180659,score_predictions_Expression_dataset_v1_mm10_C...
5,5856000,0.193556,upsample/ctloss/20250622-180659,score_predictions_Expression_dataset_v1_GRCh38...
4,3904000,0.653239,upsample/ctloss/20250622-180659,pearson_corr_genes_Expression_dataset_v1_mm10_...
3,4832000,0.668913,upsample/ctloss/20250622-180659,pearson_corr_genes_Expression_dataset_v1_GRCh3...
2,21088000,0.086916,upsample/ctloss/20250622-180659,pearson_corr_cells_Expression_dataset_v1_mm10_...
1,10304000,0.093159,upsample/ctloss/20250622-180659,pearson_corr_cells_Expression_dataset_v1_GRCh3...
0,5312000,1.085758,upsample/ctloss/20250622-180659,loss/samples/valid


In [70]:
step = 5856000
sel.query("metric in @_met and step == @step").reset_index(drop=True).sort_values("metric", ascending=False)

,step,value,run_name,metric
6,5856000,0.022320,upsample/ctloss/20250622-180659,score_predictions_Expression_dataset_v1_mm10_C...
3,5856000,0.193556,upsample/ctloss/20250622-180659,score_predictions_Expression_dataset_v1_GRCh38...
5,5856000,0.640840,upsample/ctloss/20250622-180659,pearson_corr_genes_Expression_dataset_v1_mm10_...
2,5856000,0.620366,upsample/ctloss/20250622-180659,pearson_corr_genes_Expression_dataset_v1_GRCh3...
4,5856000,0.036910,upsample/ctloss/20250622-180659,pearson_corr_cells_Expression_dataset_v1_mm10_...
1,5856000,0.035993,upsample/ctloss/20250622-180659,pearson_corr_cells_Expression_dataset_v1_GRCh3...
0,5856000,11.279546,upsample/ctloss/20250622-180659,loss/samples/valid


In [71]:
_met_sorted = [
 'score_predictions_Expression_dataset_v1_GRCh38_csv dataset/samples/valid',
 'score_predictions_Expression_dataset_v1_mm10_CPM dataset/samples/valid',
 'pearson_corr_cells_Expression_dataset_v1_GRCh38_csv dataset/samples/valid',
 'pearson_corr_cells_Expression_dataset_v1_mm10_CPM dataset/samples/valid',
 'pearson_corr_genes_Expression_dataset_v1_GRCh38_csv dataset/samples/valid',
 'pearson_corr_genes_Expression_dataset_v1_mm10_CPM dataset/samples/valid',
]

step = 5856000
print (step)
for met in _met_sorted:
	v = sel.query("metric == @met and step == @step")["value"].values[0]
	print (v, end="\t")

5856000
0.1935562342405319	0.0223200488835573	0.0359927527606487	0.0369095876812934	0.6203656792640686	0.6408404111862183	

In [62]:
_met

['loss/samples/valid',
 'pearson_corr_cells_Expression_dataset_v1_GRCh38_csv dataset/samples/valid',
 'pearson_corr_genes_Expression_dataset_v1_GRCh38_csv dataset/samples/valid',
 'score_predictions_Expression_dataset_v1_GRCh38_csv dataset/samples/valid',
 'pearson_corr_cells_Expression_dataset_v1_mm10_CPM dataset/samples/valid',
 'pearson_corr_genes_Expression_dataset_v1_mm10_CPM dataset/samples/valid',
 'score_predictions_Expression_dataset_v1_mm10_CPM dataset/samples/valid']